# Advanced Professional Baseline Notebook Structure

## ROGII - Wellbore Geology Prediction

---

# 1. Notebook Header

```python
# ============================================================
# ROGII - Wellbore Geology Prediction
# Advanced Baseline Pipeline
#
# Author: Md Ashraf
# IIT (ISM) Dhanbad
# ============================================================
```

---

In [1]:

from IPython.core import getipython
from IPython.core import getipython
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', 200)

# =============================
# PATHS
# =============================

ROOT_DIR = Path("/media/ashraf/Windows/Users/MD ASHRAF/Documents/wellbore-geology-prediction-Well-log/data/raw")

TRAIN_DIR = ROOT_DIR / "train"
TEST_DIR = ROOT_DIR / "test"

SUBMISSION_PATH = ROOT_DIR / "sample_submission.csv"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# 4. Competition Overview
print("="*60)
print("ROGII - Wellbore Geology Prediction")
print("Target: Predict TVT along horizontal wells")
print("Metric: RMSE")
print("="*60)




ROGII - Wellbore Geology Prediction
Target: Predict TVT along horizontal wells
Metric: RMSE


In [2]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def rmse(y_true, y_pred):
    """Calculate Root Mean Squared Error."""
    return np.sqrt(mean_squared_error(y_true, y_pred))

def reduce_mem_usage(df):
    """Iterate through all columns of a dataframe and modify the data type to reduce memory usage."""
    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)

    return df

## 📂 Data Loading
Extracting horizontal well data from individual files, appending identifiers, and concatenating them into our primary dataframes.

In [ ]:
# ============================================================
# 4. DATA LOADING
# ============================================================

# --- Train Data ---
train_files = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
train_data = []

for file in tqdm(train_files, desc="Loading Train Files"):
    well_name = file.stem.split("__")[0]
    df = pd.read_csv(file)
    df["WELL"] = well_name
    df["ROW_IDX"] = np.arange(len(df))
    train_data.append(df)

train_df = pd.concat(train_data, ignore_index=True)
train_df = reduce_mem_usage(train_df)
print(f"Train Shape: {train_df.shape}")

# --- Test Data ---
test_files = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
test_data = []

for file in tqdm(test_files, desc="Loading Test Files"):
    well_name = file.stem.split("__")[0]
    df = pd.read_csv(file)
    df["WELL"] = well_name
    df["ROW_IDX"] = np.arange(len(df))
    test_data.append(df)

test_df = pd.concat(test_data, ignore_index=True)
test_df = reduce_mem_usage(test_df)
print(f"Test Shape: {test_df.shape}")

Loading Train Files:   0%|          | 0/773 [00:00<?, ?it/s]

## 📊 Exploratory Data Analysis (EDA)

In [ ]:
# ============================================================
# 5. EXPLORATORY DATA ANALYSIS
# ============================================================

# Missing Values
missing = train_df.isnull().mean().sort_values(ascending=False)
display(missing[missing > 0])

# Target Distribution
fig = px.histogram(
    train_df,
    x="TVT",
    nbins=100,
    title="TVT Distribution",
    color_discrete_sequence=['#3498db']
)
fig.show()

# Well-wise Visualization
sample_well = train_df["WELL"].unique()[0]
well_df = train_df[train_df["WELL"] == sample_well]

fig = px.line(
    well_df,
    x="MD",
    y=["GR", "TVT"],
    title=f"Well Analysis: {sample_well}"
)
fig.show()

TVT_input    0.743087
GR           0.296130
ANCC         0.008961
EGFDL        0.001191
dtype: float64